# Análise de Churn de Clientes
Este notebook apresenta a análise exploratória, preparação dos dados, modelagem e impacto de negócio para o conjunto de dados Telco Customer Churn.

## Carregamento e inspeção inicial dos dados
Carregamos os dados e fazemos uma inspeção inicial para entender shape, tipos e valores ausentes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')

data_path = Path('..') / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

print('shape:', df.shape)
print('\nTipos de dados:')
print(df.dtypes)
print('\nValores ausentes por coluna:')
print(df.isnull().sum())

## Limpeza de dados
Trataremos a coluna TotalCharges convertendo para float e removendo as linhas que ficaram com valor ausente após a conversão.

In [ ]:
df_clean = df.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
missing_total = df_clean['TotalCharges'].isna().sum()
print(f'Valores missing em TotalCharges após conversão: {missing_total}')
df_clean = df_clean.dropna(subset=['TotalCharges']).reset_index(drop=True)
print('shape após limpeza:', df_clean.shape)
print(df_clean.dtypes[['TotalCharges']])

## Análise Exploratória de Dados (EDA)
Exploramos as principais relações entre churn, contrato, tenure e MonthlyCharges.

In [ ]:
output_dir = Path('..') / 'images'
output_dir.mkdir(parents=True, exist_ok=True)

churn_rate = df_clean['Churn'].value_counts(normalize=True) * 100
plt.figure(figsize=(8, 5))
sns.barplot(x=churn_rate.index, y=churn_rate.values, palette=['#4c72b0', '#dd8452'])
plt.title('Taxa geral de churn')
plt.xticks([0, 1], ['Não churn', 'Churn'])
plt.ylabel('Porcentagem (%)')
plt.tight_layout()
plt.savefig(output_dir / 'churn_rate.png')
plt.show()

### Churn por tipo de contrato
Aqui verificamos se contratos mensais, anuais ou bienais têm taxas de churn diferentes.

In [ ]:
contract_churn = df_clean.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean()).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(x=contract_churn.index, y=contract_churn.values, palette='viridis')
plt.title('Churn por tipo de contrato')
plt.ylabel('Taxa de churn')
plt.xlabel('Contract')
plt.tight_layout()
plt.savefig(output_dir / 'churn_by_contract.png')
plt.show()

### Churn por tenure
Visualizamos a distribuição de tempo de cliente entre os clientes que cancelaram.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_clean[df_clean['Churn'] == 'Yes']['tenure'], bins=20, color='#c44e52')
plt.title('Churn por tenure (meses)')
plt.xlabel('tenure')
plt.ylabel('Contagem')
plt.tight_layout()
plt.savefig(output_dir / 'churn_by_tenure.png')
plt.show()

### Churn por MonthlyCharges
Comparamos a distribuição de cobrança mensal entre clientes que churnaram e os que permaneceram.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df_clean, palette=['#4c72b0', '#dd8452'])
plt.title('MonthlyCharges por churn')
plt.xlabel('Churn')
plt.ylabel('MonthlyCharges')
plt.tight_layout()
plt.savefig(output_dir / 'monthly_charges_boxplot.png')
plt.show()

## Feature Engineering
Transformamos variáveis categóricas em variáveis dummy para alimentar o modelo de regressão logística.

In [ ]:
df_model = df_clean.copy()
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})
categorical_cols = df_model.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'customerID']
df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)
X = df_model.drop(columns=['customerID', 'Churn'])
y = df_model['Churn']
print('Número de features após encoding:', X.shape[1])

## Modelo preditivo: Regressão Logística
Aqui dividimos os dados, treinamos o modelo e avaliamos sua performance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = LogisticRegression(max_iter=200, solver='liblinear', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print('Acurácia:', accuracy_score(y_test, y_pred))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

## Business Impact
Estimamos quantos clientes estão em risco e a receita mensal potencial em risco com base nos clientes preditos como churn.

In [ ]:
df_test = df_clean.loc[y_test.index].copy()
df_test['prediction'] = y_pred
risk_customers = df_test[df_test['prediction'] == 1]
monthly_risk = risk_customers['MonthlyCharges'].mean() * len(risk_customers)

print('Clientes em risco previstos:', len(risk_customers))
print('MonthlyCharges médio dos clientes previstos como churn:', round(risk_customers['MonthlyCharges'].mean(), 2))
print('Receita mensal em risco estimada:', round(monthly_risk, 2))